In [2]:
import pandas as pd

df = pd.read_csv("cluster_output/df_com_clusters.csv")

# Acesso a computador (Q024) e internet (Q025)
# Tipo de escola
print(df["Q024"].value_counts())
print(df["Q025"].value_counts())
print(df["TP_ESCOLA"].value_counts())

Q024
A    1083934
B     781492
C     196760
D      70988
E      33669
Name: count, dtype: int64
Q025
B    1975253
A     191590
Name: count, dtype: int64
TP_ESCOLA
1    1115985
2     829530
3     221328
Name: count, dtype: int64


In [3]:
"""
Análise: acesso a tecnologia e tipo de escola por cluster.

Responde: "Em que medida o acesso a recursos tecnológicos (computador
e internet) e o tipo de escola influenciam o desempenho de cada cluster?"

Pré-requisito:
    Rodar enem_mca_final.py antes — precisa do df_com_clusters.csv

Dependências:
    pip install pandas numpy matplotlib seaborn

Uso:
    python3 enem_analise_tecnologia.py
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ---------------------------------------------------------------------------
# Configurações
# ---------------------------------------------------------------------------

OUT_DIR = Path("cluster_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COLUNAS_NOTA = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO"]
NOMES_NOTA   = ["Ciências\nNatureza", "Ciências\nHumanas", "Ling. e\nCódigos", "Matemática", "Redação"]

# Mapeamentos para labels legíveis nos gráficos
MAP_Q024 = {
    "A": "Nenhum",
    "B": "Um",
    "C": "Dois",
    "D": "Três",
    "E": "Quatro\nou mais",
}
MAP_Q025 = {"A": "Sem internet", "B": "Com internet"}
MAP_ESCOLA = {1: "Não respondeu", 2: "Pública", 3: "Privada"}

CORES_CLUSTER = sns.color_palette("Set2", 2)
RANDOM_STATE  = 42

sns.set_theme(style="whitegrid", palette="Set2")

# ---------------------------------------------------------------------------
# Carregamento
# ---------------------------------------------------------------------------

print("Carregando dados...")
df = pd.read_csv("cluster_output/df_com_clusters.csv", low_memory=False)
df["cluster"] = df["cluster"].astype(int)
df["TP_ESCOLA_LABEL"] = df["TP_ESCOLA"].map(MAP_ESCOLA)
df["Q024_LABEL"] = df["Q024"].map(MAP_Q024)
df["Q025_LABEL"] = df["Q025"].map(MAP_Q025)
print(f"  {len(df):,} linhas | clusters: {sorted(df['cluster'].unique())}")


# ---------------------------------------------------------------------------
# Análise 1 — Distribuição de Q024 (computador) por cluster
# ---------------------------------------------------------------------------

def plot_computador_por_cluster() -> None:
    """
    Mostra a proporção de cada categoria de Q024 dentro de cada cluster.
    Responde: candidatos do cluster de maior renda têm mais computadores?
    """
    print("  Gerando: distribuição de computadores por cluster...")

    ordem = ["Nenhum", "Um", "Dois", "Três", "Quatro\nou mais"]
    dist = (
        df.groupby(["cluster", "Q024_LABEL"])
        .size()
        .reset_index(name="count")
    )
    dist["pct"] = dist.groupby("cluster")["count"].transform(lambda x: x / x.sum() * 100)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for k, ax in enumerate(axes):
        data = (
            dist[dist["cluster"] == k]
            .set_index("Q024_LABEL")
            .reindex(ordem)
            .reset_index()
        )
        bars = ax.bar(data["Q024_LABEL"], data["pct"], color=CORES_CLUSTER[k], alpha=0.85)
        ax.set_title(f"Cluster {k}", fontsize=13)
        ax.set_xlabel("Computadores na residência (Q024)", fontsize=11)
        ax.set_ylabel("% de candidatos" if k == 0 else "", fontsize=11)
        ax.set_ylim(0, 55)

        # Rótulos nas barras
        for bar in bars:
            h = bar.get_height()
            if h > 0.5:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                        f"{h:.1f}%", ha="center", va="bottom", fontsize=9)

    fig.suptitle("Acesso a computador por cluster (Q024)", fontsize=14)
    plt.tight_layout()
    path = OUT_DIR / "5_computador_por_cluster.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    Salvo: {path}")


# ---------------------------------------------------------------------------
# Análise 2 — Distribuição de Q025 (internet) por cluster
# ---------------------------------------------------------------------------

def plot_internet_por_cluster() -> None:
    """
    Proporção com/sem internet dentro de cada cluster.
    """
    print("  Gerando: distribuição de internet por cluster...")

    dist = (
        df.groupby(["cluster", "Q025_LABEL"])
        .size()
        .reset_index(name="count")
    )
    dist["pct"] = dist.groupby("cluster")["count"].transform(lambda x: x / x.sum() * 100)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
    for k, ax in enumerate(axes):
        data = dist[dist["cluster"] == k].sort_values("Q025_LABEL")
        bars = ax.bar(data["Q025_LABEL"], data["pct"], color=CORES_CLUSTER[k], alpha=0.85, width=0.4)
        ax.set_title(f"Cluster {k}", fontsize=13)
        ax.set_xlabel("Acesso à internet (Q025)", fontsize=11)
        ax.set_ylabel("% de candidatos" if k == 0 else "", fontsize=11)
        ax.set_ylim(0, 105)

        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                    f"{h:.1f}%", ha="center", va="bottom", fontsize=10)

    fig.suptitle("Acesso à internet por cluster (Q025)", fontsize=14)
    plt.tight_layout()
    path = OUT_DIR / "6_internet_por_cluster.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    Salvo: {path}")


# ---------------------------------------------------------------------------
# Análise 3 — Média das notas por cluster × tipo de escola
# ---------------------------------------------------------------------------

def plot_notas_cluster_escola() -> None:
    """
    Tabela cruzada: média das notas por (cluster × tipo de escola).
    Mostra se dentro do mesmo cluster, escola pública vs privada
    ainda faz diferença no desempenho.
    """
    print("  Gerando: notas por cluster × tipo de escola...")

    # Exclui "Não respondeu"
    df_filtrado = df[df["TP_ESCOLA"] != 1].copy()

    medias = (
        df_filtrado
        .groupby(["cluster", "TP_ESCOLA_LABEL"])[COLUNAS_NOTA]
        .mean()
        .round(1)
        .reset_index()
    )

    # Um subplot por nota
    fig, axes = plt.subplots(1, len(COLUNAS_NOTA), figsize=(18, 5), sharey=False)
    escola_ordem = ["Pública", "Privada"]
    cluster_cores = {0: CORES_CLUSTER[0], 1: CORES_CLUSTER[1]}

    for ax, col, nome in zip(axes, COLUNAS_NOTA, NOMES_NOTA):
        for k in [0, 1]:
            data = (
                medias[medias["cluster"] == k]
                .set_index("TP_ESCOLA_LABEL")
                .reindex(escola_ordem)[col]
            )
            ax.plot(
                escola_ordem, data.values,
                marker="o", linewidth=2, markersize=8,
                color=cluster_cores[k], label=f"Cluster {k}",
            )
            for x, y in zip(escola_ordem, data.values):
                ax.annotate(f"{y:.0f}", (x, y),
                            textcoords="offset points", xytext=(0, 8),
                            ha="center", fontsize=9, color=cluster_cores[k])

        ax.set_title(nome, fontsize=11)
        ax.set_xlabel("")
        ax.set_ylabel("Média" if col == COLUNAS_NOTA[0] else "")
        ax.set_ylim(
            medias[col].min() - 50,
            medias[col].max() + 70,
        )

    axes[-1].legend(fontsize=10)
    fig.suptitle("Média das notas por cluster × tipo de escola", fontsize=14)
    plt.tight_layout()
    path = OUT_DIR / "7_notas_cluster_escola.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    Salvo: {path}")

    # Salva tabela em CSV também
    path_csv = OUT_DIR / "notas_cluster_escola.csv"
    medias.to_csv(path_csv, index=False)
    print(f"    Tabela salva: {path_csv}")


# ---------------------------------------------------------------------------
# Análise 4 — Média das notas por cluster × acesso à internet
# ---------------------------------------------------------------------------

def plot_notas_cluster_internet() -> None:
    """
    Média das notas por (cluster × acesso à internet).
    Mostra o efeito marginal da internet dentro de cada cluster —
    candidatos do mesmo perfil socioeconômico, com e sem internet.
    """
    print("  Gerando: notas por cluster × internet...")

    medias = (
        df.groupby(["cluster", "Q025_LABEL"])[COLUNAS_NOTA]
        .mean()
        .round(1)
        .reset_index()
    )

    fig, axes = plt.subplots(1, len(COLUNAS_NOTA), figsize=(18, 5), sharey=False)
    internet_ordem = ["Sem internet", "Com internet"]
    cluster_cores = {0: CORES_CLUSTER[0], 1: CORES_CLUSTER[1]}

    for ax, col, nome in zip(axes, COLUNAS_NOTA, NOMES_NOTA):
        for k in [0, 1]:
            data = (
                medias[medias["cluster"] == k]
                .set_index("Q025_LABEL")
                .reindex(internet_ordem)[col]
            )
            ax.plot(
                internet_ordem, data.values,
                marker="o", linewidth=2, markersize=8,
                color=cluster_cores[k], label=f"Cluster {k}",
            )
            for x, y in zip(internet_ordem, data.values):
                ax.annotate(f"{y:.0f}", (x, y),
                            textcoords="offset points", xytext=(0, 8),
                            ha="center", fontsize=9, color=cluster_cores[k])

        ax.set_title(nome, fontsize=11)
        ax.set_ylabel("Média" if col == COLUNAS_NOTA[0] else "")
        ax.set_ylim(
            medias[col].min() - 50,
            medias[col].max() + 70,
        )

    axes[-1].legend(fontsize=10)
    fig.suptitle("Média das notas por cluster × acesso à internet (Q025)", fontsize=14)
    plt.tight_layout()
    path = OUT_DIR / "8_notas_cluster_internet.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    Salvo: {path}")

    # Salva tabela
    path_csv = OUT_DIR / "notas_cluster_internet.csv"
    medias.to_csv(path_csv, index=False)
    print(f"    Tabela salva: {path_csv}")


# ---------------------------------------------------------------------------
# Análise 5 — Heatmap resumo: média das notas por cluster × computador
# ---------------------------------------------------------------------------

def plot_heatmap_notas_computador() -> None:
    """
    Heatmap da média de uma nota-síntese (média geral) por
    cluster × número de computadores. Mostra o efeito gradual
    do acesso a computador dentro de cada cluster.
    """
    print("  Gerando: heatmap notas × computador...")

    df["media_geral"] = df[COLUNAS_NOTA].mean(axis=1)
    ordem_q024 = ["Nenhum", "Um", "Dois", "Três", "Quatro\nou mais"]

    pivot = (
        df.groupby(["cluster", "Q024_LABEL"])["media_geral"]
        .mean()
        .round(1)
        .unstack("Q024_LABEL")
        .reindex(columns=ordem_q024)
    )
    pivot.index = [f"Cluster {k}" for k in pivot.index]

    fig, ax = plt.subplots(figsize=(10, 3))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".1f",
        cmap="RdYlGn",
        linewidths=0.5,
        ax=ax,
        cbar_kws={"label": "Média geral"},
        vmin=df["media_geral"].quantile(0.05),
        vmax=df["media_geral"].quantile(0.95),
    )
    ax.set_title("Média geral das notas por cluster × computadores na residência", fontsize=13)
    ax.set_xlabel("Computadores na residência (Q024)", fontsize=11)
    ax.set_ylabel("")
    plt.tight_layout()
    path = OUT_DIR / "9_heatmap_notas_computador.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"    Salvo: {path}")


# ---------------------------------------------------------------------------
# Execução
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("ANÁLISE: Tecnologia e escola por cluster")
    print("=" * 60)

    print("\n[1] Distribuição de computadores...")
    plot_computador_por_cluster()

    print("\n[2] Distribuição de internet...")
    plot_internet_por_cluster()

    print("\n[3] Notas por cluster × tipo de escola...")
    plot_notas_cluster_escola()

    print("\n[4] Notas por cluster × internet...")
    plot_notas_cluster_internet()

    print("\n[5] Heatmap notas × computador...")
    plot_heatmap_notas_computador()

    print("\n" + "=" * 60)
    print("Concluído! Imagens salvas em:", OUT_DIR.resolve())
    print("=" * 60)

Carregando dados...
  2,166,843 linhas | clusters: [np.int64(0), np.int64(1)]

ANÁLISE: Tecnologia e escola por cluster

[1] Distribuição de computadores...
  Gerando: distribuição de computadores por cluster...
    Salvo: cluster_output/5_computador_por_cluster.png

[2] Distribuição de internet...
  Gerando: distribuição de internet por cluster...
    Salvo: cluster_output/6_internet_por_cluster.png

[3] Notas por cluster × tipo de escola...
  Gerando: notas por cluster × tipo de escola...
    Salvo: cluster_output/7_notas_cluster_escola.png
    Tabela salva: cluster_output/notas_cluster_escola.csv

[4] Notas por cluster × internet...
  Gerando: notas por cluster × internet...
    Salvo: cluster_output/8_notas_cluster_internet.png
    Tabela salva: cluster_output/notas_cluster_internet.csv

[5] Heatmap notas × computador...
  Gerando: heatmap notas × computador...
    Salvo: cluster_output/9_heatmap_notas_computador.png

Concluído! Imagens salvas em: /home/sean/IA/MicrodadosEnem2023/n

In [2]:
import prince
import pandas as pd

COLUNAS_CAT_QST = [
    "Q001", "Q002", "Q003", "Q004", "Q005",
    "Q006", "Q007", "Q008", "Q009", "Q010",
    "Q011", "Q012", "Q013", "Q014", "Q015",
    "Q016", "Q017", "Q018", "Q019", "Q020",
    "Q021", "Q022", "Q023", "Q024", "Q025"
]

df = pd.read_csv("../data/df_limpo.csv", low_memory=False)
cat_df = df[COLUNAS_CAT_QST].astype(str)

# Refaz o fit (igual ao pipeline final)
sample = cat_df.sample(n=50_000, random_state=42)
mca = prince.MCA(n_components=2, random_state=42, engine="sklearn")
mca.fit(sample)

# Contribuição de cada categoria para cada componente
contrib = mca.column_contributions_.T  # linhas = componentes, colunas = categorias
print(contrib)

    Q001__A   Q001__B   Q001__C   Q001__D   Q001__E   Q001__F   Q001__G  \
0  0.004372  0.008980  0.001738  0.000411  0.000928  0.012248  0.021018   
1  0.011634  0.007949  0.000184  0.002080  0.016235  0.000009  0.013270   

    Q001__H   Q002__A   Q002__B  ...   Q022__E   Q023__A   Q023__B   Q024__A  \
0  0.001447  0.002669  0.008150  ...  0.022111  0.001306  0.012587  0.021740   
1  0.000103  0.008052  0.010858  ...  0.000148  0.000017  0.000168  0.007375   

    Q024__B   Q024__C   Q024__D   Q024__E   Q025__A   Q025__B  
0  0.002261  0.016720  0.015671  0.012961  0.012941  0.001262  
1  0.024881  0.000286  0.009605  0.028152  0.030480  0.002973  

[2 rows x 142 columns]


In [3]:
# Top 10 categorias que mais contribuem para cada componente
for comp in [0, 1]:
    print(f"\n--- Componente {comp} ---")
    print(contrib.loc[comp].sort_values(ascending=False).head(10))


--- Componente 0 ---
Q018__B    0.038444
Q021__B    0.027846
Q003__E    0.026979
Q014__A    0.026532
Q008__E    0.025208
Q010__C    0.024947
Q010__A    0.023682
Q022__E    0.022111
Q024__A    0.021740
Q001__G    0.021018
Name: 0, dtype: float64

--- Componente 1 ---
Q008__E    0.051343
Q006__Q    0.033675
Q025__A    0.030480
Q004__A    0.029800
Q019__E    0.029406
Q010__B    0.028256
Q024__E    0.028152
Q003__E    0.027591
Q003__A    0.026351
Q022__B    0.026163
Name: 1, dtype: float64


In [4]:
import pandas as pd

df = pd.read_csv("cluster_output/df_com_clusters.csv", low_memory=False)
df["cluster"] = df["cluster"].astype(int)

variaveis = {
    "Q006": "Renda familiar",
    "Q001": "Escolaridade do pai",
    "Q002": "Escolaridade da mãe",
    "Q008": "Número de banheiros",
    "Q024": "Número de computadores",
    "Q025": "Acesso à internet",
}

for col, nome in variaveis.items():
    print(f"\n{'='*60}")
    print(f"{nome} ({col})")
    print('='*60)
    dist = (
        df.groupby(["cluster", col])
        .size()
        .unstack(fill_value=0)
    )
    # Converte para proporção por cluster (%)
    dist_pct = dist.div(dist.sum(axis=1), axis=0).mul(100).round(1)
    print(dist_pct.to_string())


Renda familiar (Q006)
Q006       A     B     C     D     E    F     G     H    I    J    K    L    M    N    O    P    Q
cluster                                                                                           
0        0.4   2.4   5.6   8.9  10.6  8.3  16.5  10.2  6.8  6.1  4.8  3.2  2.8  3.4  3.3  3.0  3.8
1        8.6  43.2  21.4  12.6   6.6  3.0   2.9   0.8  0.3  0.2  0.1  0.0  0.0  0.0  0.0  0.0  0.0

Escolaridade do pai (Q001)
Q001       A     B     C     D     E     F     G     H
cluster                                               
0        0.3   3.5   6.6   8.3  35.9  21.9  18.6   5.0
1        5.7  22.7  15.7  12.5  26.9   3.3   1.1  12.1

Escolaridade da mãe (Q002)
Q002       A     B     C     D     E     F     G    H
cluster                                              
0        0.2   1.8   3.6   6.0  33.3  24.7  28.6  1.8
1        3.5  16.3  13.8  14.5  37.5   6.7   4.0  3.7

Número de banheiros (Q008)
Q008       A     B     C     D     E
cluster                 